In [ ]:
import numpy as np
import scipy.special as special
from scipy.stats import gamma, lognorm
import os
import sys

In [2]:
# Set project root relative to notebook location
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from Losses import gaussian_xgb_loss as gauss
from Losses import clayton_xgb_loss as clayton
from Losses import gumbel_xgb_loss as gumbel

In [ ]:
def clip_copula_samples(samples, eps=1e-10):
    """
    Ensure copula samples are strictly inside (0, 1)^2.
    
    Parameters:
        samples (np.ndarray): shape (n, 2), values in [0, 1]
        eps (float): small buffer to keep samples strictly in (0, 1)
    
    Returns:
        np.ndarray: clipped samples in (eps, 1 - eps)
    """
    return np.clip(samples, eps, 1 - eps)

def copula_to_gamma(copula_samples, mu1, mu2):
    """
    Transform copula samples to gamma margins with variable location parameters.
    
    Parameters:
        copula_samples (np.ndarray): shape (n, 2), values in (0, 1)
        mu1 (np.ndarray): shape (n,) parameters for Y1
        mu2 (np.ndarray): shape (n,) parameters for Y2
    
    Returns:
        np.ndarray: shape (n, 2), dependent gamma samples
    """
    u1, u2 = copula_samples[:, 0], copula_samples[:, 1]
    mu1 = np.asarray(mu1)
    mu2 = np.asarray(mu2)

    if np.any(u1 <= 0) or np.any(u2 <= 0):
        raise ValueError("Copula samples must be in (0, 1), not inclusive.")
    if np.any(mu1 <= 0) or np.any(mu2 <= 0):
        raise ValueError("Rate parameters must be > 0.")

    y1 = gamma.ppf(u1, a = 1.0, scale = mu1)
    y2 = gamma.ppf(u2, a = 1.0, scale = mu2) 

    return np.column_stack((y1, y2))

def copula_to_lognormal(copula_samples, mu1, mu2):
    """
    Transform copula samples to lognormal margins with variable location parameters.
    
    Parameters:
        copula_samples (np.ndarray): shape (n, 2), values in (0, 1)
        mu1 (np.ndarray): mean (n,) parameters for Y1
        mu2 (np.ndarray): mean (n,), rate parameters for Y2
    
    Returns:
        np.ndarray: shape (n, 2), dependent lognormal samples
    """
    u1, u2 = copula_samples[:, 0], copula_samples[:, 1]
    mu1 = np.asarray(mu1)
    mu2 = np.asarray(mu2)

    if np.any(u1 <= 0) or np.any(u2 <= 0):
        raise ValueError("Copula samples must be in (0, 1), not inclusive.")
    if np.any(mu1 <= 0) or np.any(mu2 <= 0):
        raise ValueError("Rate parameters must be > 0.")

    y1 = lognorm.ppf(u1, s=1.0, scale=mu1)
    y2 = lognorm.ppf(u2, s=1.0, scale=mu2) 

    return np.column_stack((y1, y2))

def copula_to_response(copula_samples, mu1, mu2):
    """
    Transform copula samples to lognormal margins with variable location parameters.
    
    Parameters:
        copula_samples (np.ndarray): shape (n, 2), values in (0, 1)
        mu1 (np.ndarray): mean (n,) parameters for Y1
        mu2 (np.ndarray): mean (n,), rate parameters for Y2
    
    Returns:
        np.ndarray: shape (n, 2), dependent lognormal samples
    """
    u1, u2 = copula_samples[:, 0], copula_samples[:, 1]
    mu1 = np.asarray(mu1)
    mu2 = np.asarray(mu2)

    if np.any(u1 <= 0) or np.any(u2 <= 0):
        raise ValueError("Copula samples must be in (0, 1), not inclusive.")
    if np.any(mu1 <= 0) or np.any(mu2 <= 0):
        raise ValueError("Rate parameters must be > 0.")

    y1 = lognorm.ppf(u1, s = 1.5, scale=mu1)
    y2 = gamma.ppf(u2, a = 1.5, scale = mu2/1.5) 

    return np.column_stack((y1, y2))

# shape of generated data.
n = 2000
p = 10

ratio = 0.7
seed = 2000

num_boost_round = 500

np.random.seed(seed=seed)

x = np.random.uniform(0, 1, (n, p))
x1 = x[:, 0]
x2 = x[:, 1]
x3 = x[:, 2]
x4 = x[:, 3]

mu1 = np.exp(1.5 + 2*x1 - 3*x3 + np.sin(2*np.pi*x1) + 0.5*x3**2) 
mu2 = np.exp(-1.0 + 1.5*x1 + 2*x2**2 + np.cos(2*np.pi*x2)) 

# Select one scenario for tuning
# Linear tau
#tau = special.expit(1.2*x1 - 1.5*x4)

# Non-linear tau
tau = special.expit(0.8*np.sin(2*np.pi*x1) + 1.5*(x1 > 0.5) - 1.2*(x4 > 0.7) + 0.6*x4**2)

In [ ]:
from sklearn.metrics import make_scorer
from xgboost import XGBRegressor

gauss_scorer = make_scorer(gauss.gaussian_copula_nll_metric, greater_is_better=False)

gauss_model = XGBRegressor(
    objective=gauss.gaussian_copula_nll_obj,
    n_estimators=500,
    tree_method="hist",
    eval_metric=None
)

clayton_scorer = make_scorer(clayton.clayton_copula_nll_metric, greater_is_better=False)

clayton_model = XGBRegressor(
    objective=clayton.clayton_copula_nll_obj,
    n_estimators=500,
    tree_method="hist",
    eval_metric=None
)

gumbel_scorer = make_scorer(gumbel.gumbel_copula_nll_metric, greater_is_better=False)

gumbel_model = XGBRegressor(
    objective=gumbel.gumbel_copula_nll_obj,
    n_estimators=500,
    tree_method="hist",
    eval_metric=None
)

gamma_model = XGBRegressor(
    objective="reg:gamma",
    n_estimators=500,
    tree_method="hist",
    eval_metric="gamma-nloglik"
)

normal_model = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=500,
    tree_method="hist",
    eval_metric="rmse"
)

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV

param_grid = {
    'max_depth': [1, 2, 3, 5],
    'eta':[0.01, 0.1, 0.3],
    'lambda': [0.0, 0.1, 0.2]
}

grid_search_gauss=GridSearchCV(
    estimator=gauss_model,
    param_grid=param_grid,
    scoring=gauss_scorer,
    cv=5,
    n_jobs=-1,
    verbose=2
    )

grid_search_clayton=GridSearchCV(
    estimator=clayton_model,
    param_grid=param_grid,
    scoring=clayton_scorer,
    cv=5,
    n_jobs=-1,
    verbose=2
    )

grid_search_gumbel=GridSearchCV(
    estimator=gumbel_model,
    param_grid=param_grid,
    scoring=gumbel_scorer,
    cv=5,
    n_jobs=-1,
    verbose=2
    )

grid_search_gamma=GridSearchCV(
    estimator=gamma_model,
    param_grid=param_grid,
    scoring="neg_mean_gamma_deviance",
    cv=5,
    n_jobs=-1,
    verbose=2
    )

grid_search_normal=GridSearchCV(
    estimator=normal_model,
    param_grid=param_grid,
    scoring="neg_mean_squared_error",
    cv=5,
    n_jobs=-1,
    verbose=2
    )

In [21]:
u_gauss = clip_copula_samples(gauss.sample_gaussian_copula(gauss.gauss_tau2par(tau), seed=seed), eps=1e-7)
y_gauss = copula_to_response(u_gauss, mu1, mu2)

x_main_gauss, _, y_main_gauss, _ = train_test_split(x, y_gauss, test_size=1-ratio, random_state=42)

grid_search_normal.fit(x_main_gauss, np.log(y_main_gauss[:, 0]))

print(f"Best parameters (lognormal_gauss): {grid_search_normal.best_params_}")
print(f"Best score (lognormal_gauss): {grid_search_normal.best_score_}")

Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best parameters (lognormal_gauss): {'eta': 0.1, 'lambda': 0.0, 'max_depth': 1}
Best score (lognormal_gauss): -2.456092807031742


In [22]:
grid_search_gamma.fit(x_main_gauss, y_main_gauss[:, 1])

print(f"Best parameters (gamma_gauss): {grid_search_gamma.best_params_}")
print(f"Best score (gamma_gauss): {grid_search_gamma.best_score_}")

Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best parameters (gamma_gauss): {'eta': 0.01, 'lambda': 0.2, 'max_depth': 2}
Best score (gamma_gauss): -0.7837475180555366


In [23]:
normal_model.set_params(learning_rate = grid_search_normal.best_params_["eta"], max_depth = grid_search_normal.best_params_["max_depth"], reg_lambda = grid_search_normal.best_params_["lambda"])
gamma_model.set_params(learning_rate = grid_search_gamma.best_params_["eta"], max_depth = grid_search_gamma.best_params_["max_depth"], reg_lambda = grid_search_gamma.best_params_["lambda"])

x_train_gauss, x_val_gauss, y_train_gauss, y_val_gauss = train_test_split(x_main_gauss, y_main_gauss, test_size=0.125, random_state=42)

normal_xgb1 = normal_model.fit(x_train_gauss, np.log(y_train_gauss[:,0]), eval_set=[(x_val_gauss, np.log(y_val_gauss[:,0]))])
gamma_xgb2 = gamma_model.fit(x_train_gauss, y_train_gauss[:,1], eval_set=[(x_val_gauss, y_val_gauss[:,1])])
    
mu1_main_hat = normal_xgb1.predict(x_main_gauss)
mu2_main_hat = gamma_xgb2.predict(x_main_gauss)

pseudo_obs_main_gauss = clip_copula_samples(np.column_stack((lognorm.cdf(y_main_gauss[:, 0], s = 1.5, scale=np.exp(mu1_main_hat)), gamma.cdf(y_main_gauss[:, 1], a=1.5, scale=mu2_main_hat/1.5))), eps=1e-7)

[0]	validation_0-rmse:1.69597
[1]	validation_0-rmse:1.68195
[2]	validation_0-rmse:1.67101
[3]	validation_0-rmse:1.66363
[4]	validation_0-rmse:1.65460
[5]	validation_0-rmse:1.65017
[6]	validation_0-rmse:1.64360
[7]	validation_0-rmse:1.64117
[8]	validation_0-rmse:1.63919
[9]	validation_0-rmse:1.63499
[10]	validation_0-rmse:1.63401
[11]	validation_0-rmse:1.63126
[12]	validation_0-rmse:1.63092
[13]	validation_0-rmse:1.62881
[14]	validation_0-rmse:1.62799
[15]	validation_0-rmse:1.62709
[16]	validation_0-rmse:1.62498
[17]	validation_0-rmse:1.62360
[18]	validation_0-rmse:1.62375
[19]	validation_0-rmse:1.62315
[20]	validation_0-rmse:1.61998
[21]	validation_0-rmse:1.61810
[22]	validation_0-rmse:1.61714
[23]	validation_0-rmse:1.61441
[24]	validation_0-rmse:1.61386
[25]	validation_0-rmse:1.61213
[26]	validation_0-rmse:1.60961
[27]	validation_0-rmse:1.61009
[28]	validation_0-rmse:1.60772
[29]	validation_0-rmse:1.60731
[30]	validation_0-rmse:1.60505
[31]	validation_0-rmse:1.60444
[32]	validation_0-

In [24]:
grid_search_gauss.fit(x_main_gauss, pseudo_obs_main_gauss)

print(f"Best parameters (gauss): {grid_search_gauss.best_params_}")
print(f"Best score (gauss): {grid_search_gauss.best_score_}")

Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best parameters (gauss): {'eta': 0.01, 'lambda': 0.2, 'max_depth': 2}
Best score (gauss): 319.263540427631


In [25]:
u_clayton = clip_copula_samples(clayton.sample_clayton_copula(clayton.clayton_tau2par(tau), seed=seed), eps=1e-7)
y_clayton = copula_to_response(u_clayton, mu1, mu2)

x_main_clayton, _, y_main_clayton, _ = train_test_split(x, y_clayton, test_size=1-ratio, random_state=42)

grid_search_normal.fit(x_main_clayton, np.log(y_main_clayton[:, 0]))

print(f"Best parameters (lognormal_clayton): {grid_search_normal.best_params_}")
print(f"Best score (lognormal_clayton): {grid_search_normal.best_score_}")

Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best parameters (lognormal_clayton): {'eta': 0.01, 'lambda': 0.0, 'max_depth': 1}
Best score (lognormal_clayton): -2.285293585161711


In [26]:
grid_search_gamma.fit(x_main_clayton, y_main_clayton[:, 1])

print(f"Best parameters (gamma_clayton): {grid_search_gamma.best_params_}")
print(f"Best score (gamma_clayton): {grid_search_gamma.best_score_}")

Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best parameters (gamma_clayton): {'eta': 0.01, 'lambda': 0.0, 'max_depth': 2}
Best score (gamma_clayton): -0.7676990313237243


In [27]:
normal_model.set_params(learning_rate = grid_search_normal.best_params_["eta"], max_depth = grid_search_normal.best_params_["max_depth"], reg_lambda = grid_search_normal.best_params_["lambda"])
gamma_model.set_params(learning_rate = grid_search_gamma.best_params_["eta"], max_depth = grid_search_gamma.best_params_["max_depth"], reg_lambda = grid_search_gamma.best_params_["lambda"])

x_train_clayton, x_val_clayton, y_train_clayton, y_val_clayton = train_test_split(x_main_clayton, y_main_clayton, test_size=0.125, random_state=42)

normal_xgb1 = normal_model.fit(x_train_clayton, np.log(y_train_clayton[:,0]), eval_set=[(x_val_clayton, np.log(y_val_clayton[:,0]))])
gamma_xgb2 = gamma_model.fit(x_train_clayton, y_train_clayton[:,1], eval_set=[(x_val_clayton, y_val_clayton[:,1])])
    
mu1_main_hat = normal_xgb1.predict(x_main_clayton)
mu2_main_hat = gamma_xgb2.predict(x_main_clayton)

pseudo_obs_main_clayton = clip_copula_samples(np.column_stack((lognorm.cdf(y_main_clayton[:, 0], s = 1.5, scale=np.exp(mu1_main_hat)), gamma.cdf(y_main_clayton[:, 1], a=1.5, scale=mu2_main_hat/1.5))), eps=1e-7)

[0]	validation_0-rmse:1.58433
[1]	validation_0-rmse:1.58222
[2]	validation_0-rmse:1.58016
[3]	validation_0-rmse:1.57814
[4]	validation_0-rmse:1.57616
[5]	validation_0-rmse:1.57421
[6]	validation_0-rmse:1.57231
[7]	validation_0-rmse:1.57044
[8]	validation_0-rmse:1.56861
[9]	validation_0-rmse:1.56682
[10]	validation_0-rmse:1.56506
[11]	validation_0-rmse:1.56334
[12]	validation_0-rmse:1.56165
[13]	validation_0-rmse:1.56000
[14]	validation_0-rmse:1.55838
[15]	validation_0-rmse:1.55679
[16]	validation_0-rmse:1.55523
[17]	validation_0-rmse:1.55372
[18]	validation_0-rmse:1.55222
[19]	validation_0-rmse:1.55076
[20]	validation_0-rmse:1.54933
[21]	validation_0-rmse:1.54783
[22]	validation_0-rmse:1.54645
[23]	validation_0-rmse:1.54500
[24]	validation_0-rmse:1.54367
[25]	validation_0-rmse:1.54228
[26]	validation_0-rmse:1.54100
[27]	validation_0-rmse:1.53965
[28]	validation_0-rmse:1.53842
[29]	validation_0-rmse:1.53713
[30]	validation_0-rmse:1.53594
[31]	validation_0-rmse:1.53469
[32]	validation_0-

In [28]:
grid_search_clayton.fit(x_main_clayton, pseudo_obs_main_clayton)

print(f"Best parameters (clayton): {grid_search_clayton.best_params_}")
print(f"Best score (clayton): {grid_search_clayton.best_score_}")

Fitting 5 folds for each of 36 candidates, totalling 180 fits


c:\Users\antho\master-thesis-project\xgb_env\Lib\site-packages\sklearn\model_selection\_search.py:1137: UserWarning: One or more of the test scores are non-finite: [315.97726328 308.43549784 294.55779221          nan 316.01367109
 309.23426002 294.96920944          nan 316.04839937 310.08455417
 294.50096163          nan 310.89983145          nan          nan
          nan 311.08137725          nan          nan          nan
 311.10996095          nan          nan          nan 287.85304646
          nan          nan          nan 289.76404571          nan
          nan          nan 288.77560311          nan          nan
          nan]
  warnings.warn(


Best parameters (clayton): {'eta': 0.01, 'lambda': 0.2, 'max_depth': 1}
Best score (clayton): 316.04839937094414


In [29]:
u_gumbel = clip_copula_samples(gumbel.sample_gumbel_copula(gumbel.gumbel_tau2par(tau), seed=seed), eps=1e-7)
y_gumbel = copula_to_response(u_gumbel, mu1, mu2)

x_main_gumbel, _, y_main_gumbel, _ = train_test_split(x, y_gumbel, test_size=1-ratio, random_state=42)

grid_search_normal.fit(x_main_gumbel, np.log(y_main_gumbel[:, 0]))

print(f"Best parameters (lognormal_gumbel): {grid_search_normal.best_params_}")
print(f"Best score (lognormal_gumbel): {grid_search_normal.best_score_}")

Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best parameters (lognormal_gumbel): {'eta': 0.01, 'lambda': 0.0, 'max_depth': 2}
Best score (lognormal_gumbel): -2.3050029610357226


In [30]:
grid_search_gamma.fit(x_main_gumbel, y_main_gumbel[:, 1])

print(f"Best parameters (gamma_gumbel): {grid_search_gamma.best_params_}")
print(f"Best score (gamma_gumbel): {grid_search_gamma.best_score_}")

Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best parameters (gamma_gumbel): {'eta': 0.01, 'lambda': 0.0, 'max_depth': 2}
Best score (gamma_gumbel): -0.7826235336506947


In [31]:
normal_model.set_params(learning_rate = grid_search_normal.best_params_["eta"], max_depth = grid_search_normal.best_params_["max_depth"], reg_lambda = grid_search_normal.best_params_["lambda"])
gamma_model.set_params(learning_rate = grid_search_gamma.best_params_["eta"], max_depth = grid_search_gamma.best_params_["max_depth"], reg_lambda = grid_search_gamma.best_params_["lambda"])

x_train_gumbel, x_val_gumbel, y_train_gumbel, y_val_gumbel = train_test_split(x_main_gumbel, y_main_gumbel, test_size=0.125, random_state=42)

normal_xgb1 = normal_model.fit(x_train_gumbel, np.log(y_train_gumbel[:,0]), eval_set=[(x_val_gumbel, np.log(y_val_gumbel[:,0]))])
gamma_xgb2 = gamma_model.fit(x_train_gumbel, y_train_gumbel[:,1], eval_set=[(x_val_gumbel, y_val_gumbel[:,1])])
    
mu1_main_hat = normal_xgb1.predict(x_main_gumbel)
mu2_main_hat = gamma_xgb2.predict(x_main_gumbel)

pseudo_obs_main_gumbel = clip_copula_samples(np.column_stack((lognorm.cdf(y_main_gumbel[:, 0], s = 1.5, scale=np.exp(mu1_main_hat)), gamma.cdf(y_main_gumbel[:, 1], a=1.5, scale=mu2_main_hat/1.5))), eps=1e-7)

[0]	validation_0-rmse:1.64546
[1]	validation_0-rmse:1.64226
[2]	validation_0-rmse:1.63914
[3]	validation_0-rmse:1.63608
[4]	validation_0-rmse:1.63308
[5]	validation_0-rmse:1.63014
[6]	validation_0-rmse:1.62747
[7]	validation_0-rmse:1.62466
[8]	validation_0-rmse:1.62210
[9]	validation_0-rmse:1.61939
[10]	validation_0-rmse:1.61694
[11]	validation_0-rmse:1.61434
[12]	validation_0-rmse:1.61199
[13]	validation_0-rmse:1.60950
[14]	validation_0-rmse:1.60725
[15]	validation_0-rmse:1.60486
[16]	validation_0-rmse:1.60271
[17]	validation_0-rmse:1.60042
[18]	validation_0-rmse:1.59836
[19]	validation_0-rmse:1.59616
[20]	validation_0-rmse:1.59418
[21]	validation_0-rmse:1.59208
[22]	validation_0-rmse:1.59019
[23]	validation_0-rmse:1.58817
[24]	validation_0-rmse:1.58636
[25]	validation_0-rmse:1.58443
[26]	validation_0-rmse:1.58270
[27]	validation_0-rmse:1.58085
[28]	validation_0-rmse:1.57920
[29]	validation_0-rmse:1.57742
[30]	validation_0-rmse:1.57585
[31]	validation_0-rmse:1.57415
[32]	validation_0-

In [32]:
grid_search_gumbel.fit(x_main_gumbel, pseudo_obs_main_gumbel)

print(f"Best parameters (gumbel): {grid_search_gumbel.best_params_}")
print(f"Best score (gumbel): {grid_search_gumbel.best_score_}")

Fitting 5 folds for each of 36 candidates, totalling 180 fits


c:\Users\antho\master-thesis-project\xgb_env\Lib\site-packages\sklearn\model_selection\_search.py:1137: UserWarning: One or more of the test scores are non-finite: [304.37987851 302.3169511  295.18223561 261.57677407 304.37900343
 302.40404122 294.81442643 258.8469772  304.37803883 302.46246742
 294.74540512 260.75557181 304.30065426          nan          nan
          nan 304.28488708          nan          nan          nan
 304.37676329          nan          nan          nan 289.78803168
          nan          nan          nan 289.36535029          nan
          nan          nan 289.78635576          nan          nan
          nan]
  warnings.warn(


Best parameters (gumbel): {'eta': 0.01, 'lambda': 0.0, 'max_depth': 1}
Best score (gumbel): 304.37987851053236
